In [ ]:
import numpy as np
import h5py
import pandas as pd
import matplotlib.pyplot as plt
from astropy.convolution import convolve, Gaussian1DKernel, Box1DKernel
from scipy.signal import argrelmin, argrelmax

from pipeline.processing.remove_spikes import remove_spikes
from pipeline.processing.fit_continuum import fit_continuum
from pipeline.processing.get_absorption_intervals import get_absorption_intervals
from pipeline.processing.fit_gaussians import fit_single_gaussian, fit_two_gaussians, ftest_which_gaussian
from pipeline.processing.measure_line import measure_line
from pipeline.processing.scale_civ_to_nv import scale_civ_to_nv
from pipeline.processing.utility_functions import *
from pipeline.processing.parameters import CIV_CONTINUUM_WINDOWS, CIV_FIT_WINDOW, LYA_NV_FIT_WINDOW, NV_LYA_CONTINUUM_WINDOWS, CIV_AIR, NV_AIR, LYA_AIR, BAL_MIN_WIDTH
from utility.get_intercepts_near_line import get_intercepts_near_line

settings = {
        'font.size':16,
        # 'xtick.direction':'in', 
        # 'xtick.minor.visible':True,
        # 'xtick.top':True,
        # 'ytick.direction':'in', 
        # 'ytick.minor.visible':True,
        # 'ytick.right':True,
        # 'xtick.major.size':4.0,
        # 'xtick.minor.size':2.0,
        # 'ytick.major.size':4.0,
        # 'ytick.minor.size':2.0,
        'legend.frameon':False,
        # 'mathtext.fontset':'stix',
        # 'font.family':'STIXGeneral'
    }

plt.rcParams.update(**settings) 

RELEVANT_RESULTS_PATH = "data/relevant_results.csv"
FIT_PARAMS_PATH = "data/fit_params.csv"

In [2]:
results = pd.read_csv(RELEVANT_RESULTS_PATH,index_col=0,dtype={"targetid":"str"})
results.info()

<class 'pandas.core.frame.DataFrame'>
Index: 186890 entries, 39628261600268308 to 39627768828270015
Data columns (total 32 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   rew_civ                  186890 non-null  float64
 1   rew_nv                   186890 non-null  float64
 2   rew_lya                  186890 non-null  float64
 3   snr_1700A                186888 non-null  float64
 4   snr_civ                  186890 non-null  float64
 5   snr_nv                   186890 non-null  float64
 6   snr_lya                  186890 non-null  float64
 7   pixel_used_civ           186890 non-null  float64
 8   pixel_used_blue_end_lya  186890 non-null  float64
 9   continuum_redchi         186890 non-null  float64
 10  z                        186890 non-null  float64
 11  zerr                     186890 non-null  float64
 12  civ_1549_flux            186890 non-null  float64
 13  civ_1549_flux_ivar       186890 non-n